# S2. Описова аналітика повних даних TeleNarratives

**Що робить ноутбук:** читає з вашого Google Drive файли, які завантажив S1 (`MyDrive/narratives-thesis/data/raw/`), рахує таблиці
і графіки EDA (цілісність, канали, місяці, мова, слабкі мітки, перетин із золотим набором, пересилання, дублікати) і зберігає
їх у `MyDrive/narratives-thesis/results/02_eda/`. Код репозиторію вбудований у ноутбук, окремо нічого ставити не потрібно.
Золотий набір (412 повідомлень) завантажується з GitHub-репозиторію TeleNarratives.

**Що зробити вам:** Runtime → Run all (GPU не потрібен, достатньо звичайного середовища). Дозвольте доступ до Drive.
Орієнтовний час 10-20 хвилин (найдовше визначення мови). Потрібно приблизно 6-8 ГБ оперативної пам'яті.
Результати я прочитаю з Drive сам. Архів `02_eda.zip` у тій самій папці можна додати до репозиторію.


In [ ]:
# 1. Drive, залежності, шляхи
import hashlib, json, os, subprocess, sys, time
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/narratives-thesis")
    REPO = Path("/content/repo")
except ImportError:  # запуск поза Colab (тестовий)
    BASE = Path("./narratives-thesis").resolve()
    REPO = Path("./repo").resolve()

RAW = BASE / "data" / "raw"
OUT = BASE / "results" / "02_eda"
assert RAW.exists(), f"Немає {RAW}. Спершу запустіть ноутбук S1."
OUT.mkdir(parents=True, exist_ok=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "langid"], check=True)
print("Дані:", RAW)
print("Результати:", OUT)


In [ ]:
# 2. Код репозиторію (вбудований у ноутбук)
FILES = {'src/narr/__init__.py': '"""Код магістерської роботи: гібридні моделі (граф + текст) для виявлення дезінформаційних наративів."""\n\n__version__ = "0.1.0"\n', 'src/narr/eda.py': '"""S2: описова аналітика повних даних TeleNarratives.\n\nМодуль самодостатній (pandas, numpy, pyarrow, matplotlib), тому той самий код працює в Colab і в тестах.\nУсі функції приймають таблиці як DataFrame і не читають диск, крім `load_inputs`.\n"""\nfrom __future__ import annotations\n\nimport json\nimport re\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nKEY_MULT = 10_000_000          # ключ повідомлення = channel_id * KEY_MULT + message_id\nDATE_MIN, DATE_MAX = "2024-12-16", "2026-02-27"\n# Кількості з README і MANIFEST.json графового набору (знімок 11.03.2026)\nREADME_COUNTS = {"channels": 111, "messages": 1_052_563, "forward_edges": 35_697, "semantic_edges": 52_414,\n                 "channel_flow_edges": 214, "sem_channel_flow_edges": 1_273}\nGOLD_COLS = {"text": "text", "channel": "channel_id", "message": "message_id", "date": "date_utc",\n             "binary": "Binary label", "level1": "General label (level-1)"}\nMSG_COLS = ["channel_id", "message_id", "date_utc", "channel_username", "text", "views", "forwards", "has_media",\n            "binary_label", "is_labeled_batch", "narrative_id", "sub_narrative_id", "label_confidence"]\nTG_COLS = ["channel_id", "message_id", "is_reply", "reply_to_msg_id", "is_forwarded", "fwd_from_channel_id",\n           "fwd_from_message_id", "date_utc", "views", "forwards", "replies_count", "has_media"]\n\n_NON_WORD = re.compile(r"[^\\w\\s]+", re.UNICODE)\n\n\n# ---------------------------------------------------------------- допоміжні\n\ndef keys(df: pd.DataFrame, ch: str = "channel_id", msg: str = "message_id") -> pd.Series:\n    return df[ch].astype("int64") * KEY_MULT + df[msg].astype("int64")\n\n\ndef _flag(s: pd.Series) -> pd.Series:\n    """Булева маска з порожніми значеннями як False (працює для bool, object з None, nullable boolean)."""\n    return s.eq(True).fillna(False).astype(bool)\n\n\ndef normalize_text(s: pd.Series) -> pd.Series:\n    out = s.fillna("").astype(str).str.lower().str.replace(_NON_WORD, "", regex=True)\n    return out.str.replace(r"\\s+", " ", regex=True).str.strip()\n\n\ndef _share(a, b) -> float:\n    return float(a) / float(b) if b else float("nan")\n\n\ndef _binary_scores(y: np.ndarray, p: np.ndarray) -> dict:\n    y, p = np.asarray(y, bool), np.asarray(p, bool)\n    tp, fp = int((y & p).sum()), int((~y & p).sum())\n    fn, tn = int((y & ~p).sum()), int((~y & ~p).sum())\n\n    def f1(t, f_p, f_n):\n        d = 2 * t + f_p + f_n\n        return 2 * t / d if d else float("nan")\n\n    den = np.sqrt(float(tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))\n    return {"n": int(len(y)), "tp": tp, "fp": fp, "fn": fn, "tn": tn,\n            "accuracy": _share(tp + tn, len(y)), "precision": _share(tp, tp + fp), "recall": _share(tp, tp + fn),\n            "f1_pos": f1(tp, fp, fn), "macro_f1": (f1(tp, fp, fn) + f1(tn, fn, fp)) / 2,\n            "mcc": (tp * tn - fp * fn) / den if den else float("nan")}\n\n\n# ---------------------------------------------------------------- читання\n\ndef load_inputs(raw_dir: Path | str, golden_path: Path | str) -> dict:\n    """Читає потрібні таблиці з data/raw (повні parquet не тримаємо в пам\'яті, якщо вони не потрібні)."""\n    import pyarrow.parquet as pq\n\n    raw_dir = Path(raw_dir)\n    mpath = next(iter(sorted(raw_dir.rglob("messages.parquet"))), None)\n    if mpath is None:\n        raise FileNotFoundError(f"не знайдено messages.parquet у {raw_dir}")\n    gd = mpath.parent\n    tg_path = raw_dir / "telegram_messages.parquet"\n\n    msgs = pq.read_table(mpath, columns=MSG_COLS).to_pandas()\n    tg = pq.read_table(tg_path, columns=TG_COLS).to_pandas()\n    gold = pd.read_csv(golden_path)\n\n    gold_ids = [int(x) for x in gold[GOLD_COLS["message"]].unique()]\n    tg_text = pq.read_table(tg_path, columns=["channel_id", "message_id", "text"],\n                            filters=[("message_id", "in", gold_ids)]).to_pandas()\n\n    fwd = pq.read_table(gd / "forward_edges.parquet",\n                        columns=["logical_src_channel_id", "logical_src_message_id",\n                                 "logical_dst_channel_id", "logical_dst_message_id"]).to_pandas()\n    sem = pq.read_table(gd / "semantic_edges.parquet",\n                        columns=["src_channel_id", "src_message_id", "dst_channel_id", "dst_message_id",\n                                 "similarity_score"]).to_pandas()\n    return {"msgs": msgs, "tg": tg, "tg_gold_text": tg_text, "gold": gold, "fwd": fwd, "sem": sem,\n            "channels": pq.read_table(gd / "channels.parquet").to_pandas(),\n            "coding": pd.read_csv(gd / "narrative_subnarrative_coding_en.csv"),\n            "paths": {"messages": str(mpath), "telegram_messages": str(tg_path)}}\n\n\n# ---------------------------------------------------------------- таблиці\n\ndef check_integrity(msgs: pd.DataFrame, tg: pd.DataFrame, date_min: str = DATE_MIN, date_max: str = DATE_MAX) -> dict:\n    km, kt = keys(msgs).to_numpy(), keys(tg).to_numpy()\n    both = np.intersect1d(km, kt)\n    lo = pd.Timestamp(date_min, tz="UTC")\n    hi = pd.Timestamp(date_max, tz="UTC") + pd.Timedelta(days=1)\n    out = {"n_msgs": int(len(msgs)), "n_tg": int(len(tg)),\n           "n_channels_msgs": int(msgs["channel_id"].nunique()), "n_channels_tg": int(tg["channel_id"].nunique()),\n           "dup_keys_msgs": int(pd.Series(km).duplicated().sum()), "dup_keys_tg": int(pd.Series(kt).duplicated().sum()),\n           "keys_both": int(len(both)), "keys_only_msgs": int(len(np.unique(km)) - len(both)),\n           "keys_only_tg": int(len(np.unique(kt)) - len(both))}\n    for name, df in (("msgs", msgs), ("tg", tg)):\n        d = pd.to_datetime(df["date_utc"], utc=True, errors="coerce")\n        out[f"unparsed_dates_{name}"] = int(d.isna().sum())\n        out[f"date_min_{name}"] = str(d.min())\n        out[f"date_max_{name}"] = str(d.max())\n        out[f"out_of_range_{name}"] = int(((d < lo) | (d >= hi)).sum())\n    return out\n\n\ndef monthly_table(msgs: pd.DataFrame) -> pd.DataFrame:\n    d = pd.to_datetime(msgs["date_utc"], utc=True)\n    lab = _flag(msgs["is_labeled_batch"])\n    t = pd.DataFrame({"month": d.dt.strftime("%Y-%m"), "n": 1, "n_labeled": lab.astype(int),\n                      "n_narrative": (lab & msgs["narrative_id"].notna()).astype(int)})\n    t = t.groupby("month", as_index=False).sum()\n    t["narrative_share_of_labeled"] = t["n_narrative"] / t["n_labeled"].where(t["n_labeled"] > 0)\n    t["partial_month"] = t["month"].isin([t["month"].min(), t["month"].max()])\n    return t\n\n\ndef per_channel(msgs: pd.DataFrame, tg: pd.DataFrame, channels: pd.DataFrame) -> pd.DataFrame:\n    lab = _flag(msgs["is_labeled_batch"])\n    g = pd.DataFrame({"channel_id": msgs["channel_id"], "n_messages": 1, "n_labeled": lab.astype(int),\n                      "n_narrative": (lab & msgs["narrative_id"].notna()).astype(int),\n                      "length": msgs["text"].fillna("").str.len()})\n    agg = g.groupby("channel_id").agg(n_messages=("n_messages", "sum"), n_labeled=("n_labeled", "sum"),\n                                      n_narrative=("n_narrative", "sum"), median_length=("length", "median")).reset_index()\n    t2 = tg.groupby("channel_id").agg(forwarded_share=("is_forwarded", "mean"), reply_share=("is_reply", "mean")).reset_index()\n    ch = channels[["channel_id", "username", "narrative_presence", "narrative_presence_status", "channel_label",\n                   "labeled_messages", "narrative_messages", "narrative_share"]].rename(columns={\n        "narrative_presence": "table_narrative_presence", "narrative_presence_status": "table_status",\n        "channel_label": "table_label", "labeled_messages": "table_labeled", "narrative_messages": "table_narrative",\n        "narrative_share": "table_share"})\n    out = ch.merge(agg, on="channel_id", how="outer").merge(t2, on="channel_id", how="left")\n    for c in ("n_messages", "n_labeled", "n_narrative"):\n        out[c] = out[c].fillna(0).astype(int)\n    out["share_recomputed"] = out["n_narrative"] / out["n_labeled"].where(out["n_labeled"] > 0)\n    out["counts_match"] = (out["n_labeled"] == out["table_labeled"]) & (out["n_narrative"] == out["table_narrative"])\n    return out.sort_values("n_messages", ascending=False).reset_index(drop=True)\n\n\ndef label_tables(msgs: pd.DataFrame, coding: pd.DataFrame) -> tuple[dict, dict[str, pd.DataFrame]]:\n    lab = _flag(msgs["is_labeled_batch"])\n    narr = msgs["narrative_id"].notna()\n    n_lab, n_narr = int(lab.sum()), int((lab & narr).sum())\n    summary = {"n_messages": int(len(msgs)), "n_labeled": n_lab, "share_labeled": _share(n_lab, len(msgs)),\n               "n_narrative_any": int(narr.sum()), "n_narrative_labeled": n_narr,\n               "narrative_not_labeled": int((narr & ~lab).sum()),\n               "narrative_share_of_labeled": _share(n_narr, n_lab),\n               "sub_without_narrative": int((msgs["sub_narrative_id"].notna() & ~narr).sum()),\n               "confidence_missing_among_labeled": int(msgs.loc[lab, "label_confidence"].isna().sum())}\n\n    # consistency of sub_narrative_id -> narrative_id with the coding file\n    sub2narr = coding.drop_duplicates("sub_narrative_id").set_index("sub_narrative_id")["narrative_id"]\n    both = msgs.loc[msgs["sub_narrative_id"].notna() & narr, ["sub_narrative_id", "narrative_id"]]\n    mapped = both["sub_narrative_id"].astype("int64").map(sub2narr)\n    summary["sub_narrative_unknown_in_coding"] = int(mapped.isna().sum())\n    summary["sub_to_narrative_match_share"] = _share(int((mapped == both["narrative_id"]).sum()), len(both))\n    summary["max_sub_narrative_id"] = int(msgs["sub_narrative_id"].max()) if msgs["sub_narrative_id"].notna().any() else None\n    summary["sub_ids_not_in_coding"] = int(msgs["sub_narrative_id"].dropna().astype("int64").map(sub2narr).isna().sum())\n\n    # confidence of labels\n    conf = pd.DataFrame({"label_confidence": msgs.loc[lab, "label_confidence"], "narrative": narr[lab]})\n    conf_t = (conf.groupby("label_confidence")["narrative"].agg(n="size", n_narrative="sum").reset_index()\n              .assign(n_not_narrative=lambda d: d["n"] - d["n_narrative"]))\n    conf_t["n_narrative"] = conf_t["n_narrative"].astype(int)\n\n    # binary_label vs labeled batch\n    bl = msgs["binary_label"].astype(object).where(msgs["binary_label"].notna(), "(порожньо)").astype(str)\n    cross = (pd.DataFrame({"binary_label": bl, "is_labeled_batch": lab, "has_narrative_id": narr})\n             .groupby(["binary_label", "is_labeled_batch", "has_narrative_id"]).size().rename("n").reset_index())\n\n    # meta-narratives of weak labels (through narrative_id, not sub_narrative_id)\n    n2m = coding.drop_duplicates("narrative_id")[["narrative_id", "level_1_id", "level_1"]]\n    summary["narrative_to_meta_unique"] = bool((coding.groupby("narrative_id")["level_1_id"].nunique() == 1).all())\n    m = msgs.loc[lab & narr, ["narrative_id"]].merge(n2m, on="narrative_id", how="left")\n    meta = (m.groupby(["level_1_id", "level_1"], dropna=False).size().rename("n_weak").reset_index()\n            .sort_values("n_weak", ascending=False))\n    meta["share_weak"] = meta["n_weak"] / meta["n_weak"].sum()\n\n    # text length\n    L = msgs["text"].fillna("").str.len()\n    groups = {"усі": pd.Series(True, index=msgs.index), "розмічені": lab, "розмічені, наративні": lab & narr,\n              "розмічені, ненаративні": lab & ~narr, "нерозмічені": ~lab}\n    rows = []\n    for name, mask in groups.items():\n        x = L[mask]\n        rows.append({"group": name, "n": int(len(x)), "mean": float(x.mean()), "p10": float(x.quantile(.1)),\n                     "median": float(x.median()), "p90": float(x.quantile(.9)), "max": int(x.max())})\n    return summary, {"label_confidence": conf_t, "binary_label_crosstab": cross, "meta_weak": meta,\n                     "length_stats": pd.DataFrame(rows)}\n\n\ndef language_tables(msgs: pd.DataFrame, gold: pd.DataFrame, lang_fn, n_unlabeled: int, seed: int,\n                    gcols: dict = GOLD_COLS) -> dict[str, pd.DataFrame]:\n    """Мова всіх розмічених + випадкової вибірки нерозмічених + золотих. lang_fn(Series) -> масив кодів."""\n    lab = _flag(msgs["is_labeled_batch"]).to_numpy()\n    rng = np.random.default_rng(seed)\n    idx_lab = np.flatnonzero(lab)\n    idx_unl = np.flatnonzero(~lab)\n    take = np.sort(rng.choice(idx_unl, size=min(n_unlabeled, len(idx_unl)), replace=False)) if len(idx_unl) else idx_unl\n    sub = msgs.iloc[np.concatenate([idx_lab, take])][["channel_id", "message_id", "text", "narrative_id"]].copy()\n    sub["is_labeled"] = np.r_[np.ones(len(idx_lab), bool), np.zeros(len(take), bool)]\n    sub["narrative"] = sub["narrative_id"].notna() & sub["is_labeled"]\n    sub["lang"] = lang_fn(sub["text"].fillna("").astype(str).str.slice(0, 300))\n\n    by_group = (sub.assign(group=np.where(sub["is_labeled"], "розмічені (усі)", "нерозмічені (вибірка)"))\n                .groupby(["group", "lang"]).size().rename("n").reset_index())\n    by_group["share"] = by_group["n"] / by_group.groupby("group")["n"].transform("sum")\n\n    wl = sub[sub["is_labeled"]]\n    x_label = (wl.groupby("lang")["narrative"].agg(n="size", n_narrative="sum").reset_index()\n               .assign(source="слабкі мітки (розмічені)"))\n\n    g = gold.copy()\n    g["lang"] = lang_fn(g[gcols["text"]].fillna("").astype(str).str.slice(0, 300))\n    gx = (g.groupby("lang")[gcols["binary"]].agg(n="size", n_narrative="sum").reset_index()\n          .assign(source="золота розмітка"))\n    x_label = pd.concat([x_label, gx], ignore_index=True)\n    x_label["n_narrative"] = x_label["n_narrative"].astype(int)\n    x_label["narrative_share"] = x_label["n_narrative"] / x_label["n"]\n\n    ch = sub.groupby(["channel_id", "lang"]).size().unstack(fill_value=0)\n    ch["n_language_sample"] = ch.sum(axis=1)\n    for c in [c for c in ch.columns if c != "n_language_sample"]:\n        ch[f"share_{c}"] = ch[c] / ch["n_language_sample"]\n    return {"language_by_group": by_group, "language_x_label": x_label, "language_by_channel": ch.reset_index()}\n\n\ndef graph_membership(msgs: pd.DataFrame, fwd: pd.DataFrame, sem: pd.DataFrame) -> pd.DataFrame:\n    km = keys(msgs).to_numpy()\n    f = np.union1d(keys(fwd, "logical_src_channel_id", "logical_src_message_id").to_numpy(),\n                   keys(fwd, "logical_dst_channel_id", "logical_dst_message_id").to_numpy())\n    s = np.union1d(keys(sem, "src_channel_id", "src_message_id").to_numpy(),\n                   keys(sem, "dst_channel_id", "dst_message_id").to_numpy())\n    in_f, in_s = np.isin(km, f), np.isin(km, s)\n    lab = _flag(msgs["is_labeled_batch"]).to_numpy()\n    narr = (lab & msgs["narrative_id"].notna().to_numpy())\n    grp = np.where(in_f & in_s, "forward і semantic", np.where(in_f, "лише forward",\n                                                              np.where(in_s, "лише semantic", "поза графом")))\n    t = (pd.DataFrame({"group": grp, "n": 1, "n_labeled": lab.astype(int), "n_narrative": narr.astype(int)})\n         .groupby("group", as_index=False).sum())\n    t["labeled_share"] = t["n_labeled"] / t["n"]\n    t["narrative_share_of_labeled"] = t["n_narrative"] / t["n_labeled"].where(t["n_labeled"] > 0)\n    return t\n\n\ndef forwards_summary(tg: pd.DataFrame, msgs: pd.DataFrame, channels: pd.DataFrame, n_forward_edges: int) -> dict:\n    fw = tg[_flag(tg["is_forwarded"])]\n    corpus = set(channels["channel_id"].astype("int64")) | set(msgs["channel_id"].astype("int64"))\n    src = fw["fwd_from_channel_id"].dropna().astype("int64")\n    in_corpus_channel = src.isin(corpus)\n    ok = fw["fwd_from_channel_id"].notna() & fw["fwd_from_message_id"].notna()\n    ofk = (fw.loc[ok, "fwd_from_channel_id"].astype("int64") * KEY_MULT + fw.loc[ok, "fwd_from_message_id"].astype("int64"))\n    orig_present = ofk.isin(set(keys(tg)))\n    return {"n_forwarded": int(len(fw)), "forwarded_share": _share(len(fw), len(tg)),\n            "n_unique_source_channels": int(src.nunique()),\n            "n_forwarded_from_corpus_channel": int(in_corpus_channel.sum()),\n            "n_forwarded_with_original_in_corpus": int(orig_present.sum()),\n            "n_forward_edges_file": int(n_forward_edges),\n            "n_replies": int(_flag(tg["is_reply"]).sum()), "reply_share": _share(int(_flag(tg["is_reply"]).sum()), len(tg))}\n\n\ndef _gold_merge(gold, msgs, gcols):\n    g = gold.copy()\n    g["_key"] = keys(g, gcols["channel"], gcols["message"])\n    m = msgs[["channel_id", "message_id", "text", "is_labeled_batch", "narrative_id", "label_confidence"]].copy()\n    m["_key"] = keys(m)\n    return g.merge(m[["_key", "text", "is_labeled_batch", "narrative_id", "label_confidence"]].rename(\n        columns={"text": "text_graph"}), on="_key", how="left")\n\n\ndef gold_overlap(gold: pd.DataFrame, msgs: pd.DataFrame, coding: pd.DataFrame, tg_gold_text: pd.DataFrame,\n                 fwd: pd.DataFrame, sem: pd.DataFrame, gcols: dict = GOLD_COLS) -> tuple[dict, dict[str, pd.DataFrame]]:\n    m = _gold_merge(gold, msgs, gcols)\n    present = m["text_graph"].notna()\n    lab = _flag(m["is_labeled_batch"]) & present\n    y = m[gcols["binary"]].astype(bool)\n    summary = {"n_gold": int(len(m)), "n_gold_in_graph_messages": int(present.sum()),\n               "n_gold_labeled_weak": int(lab.sum()),\n               "n_gold_present_not_labeled": int((present & ~lab).sum()),\n               "gold_narrative_share_present_not_labeled": _share(int((y & present & ~lab).sum()), int((present & ~lab).sum()))}\n    km_gold = keys(gold, gcols["channel"], gcols["message"])\n    f = np.union1d(keys(fwd, "logical_src_channel_id", "logical_src_message_id").to_numpy(),\n                   keys(fwd, "logical_dst_channel_id", "logical_dst_message_id").to_numpy())\n    s = np.union1d(keys(sem, "src_channel_id", "src_message_id").to_numpy(),\n                   keys(sem, "dst_channel_id", "dst_message_id").to_numpy())\n    summary["n_gold_in_forward_edges"] = int(np.isin(km_gold.to_numpy(), f).sum())\n    summary["n_gold_in_semantic_edges"] = int(np.isin(km_gold.to_numpy(), s).sum())\n\n    tables: dict[str, pd.DataFrame] = {}\n    if lab.any():\n        weak = m.loc[lab, "narrative_id"].notna().to_numpy()\n        sc = _binary_scores(y[lab].to_numpy(), weak)\n        summary["weak_vs_gold"] = sc\n        tables["weak_vs_gold_confusion"] = pd.DataFrame(\n            [[sc["tn"], sc["fp"]], [sc["fn"], sc["tp"]]], index=["золота: ні", "золота: так"],\n            columns=["слабка: ні", "слабка: так"]).reset_index().rename(columns={"index": ""})\n        # мета-наративи: назви рівня 1 у золотому наборі проти слабких міток\n        n2m = coding.drop_duplicates("narrative_id").set_index("narrative_id")["level_1"]\n        both = lab & y & m["narrative_id"].notna()\n        a = m.loc[both, gcols["level1"]].astype(str)\n        b = m.loc[both, "narrative_id"].astype("int64").map(n2m).astype(str)\n        summary["meta_pairs_both_narrative"] = int(both.sum())\n        summary["meta_match_share"] = _share(int((a.to_numpy() == b.to_numpy()).sum()), int(both.sum()))\n        summary["gold_meta_names_not_in_coding"] = int((~a.isin(set(coding["level_1"]))).sum())\n        tables["weak_vs_gold_meta"] = pd.crosstab(a.rename("золотий мета-наратив"), b.rename("слабкий мета-наратив")).reset_index()\n    # варіанти тексту\n    tt = tg_gold_text.copy()\n    tt["_key"] = keys(tt)\n    mm = m.merge(tt[["_key", "text"]].rename(columns={"text": "text_tg"}), on="_key", how="left")\n    gt = mm[gcols["text"]].fillna("").astype(str)\n    gn = normalize_text(gt)\n    rows = []\n    for name, col in (("messages.parquet (сирий)", "text_graph"), ("telegram_messages.parquet (нормалізований)", "text_tg")):\n        has = mm[col].notna()\n        other = mm.loc[has, col].astype(str)\n        on = normalize_text(other)\n        gnn = gn[has]\n        rows.append({"варіант тексту": name, "n_знайдено": int(has.sum()),\n                     "точно збігається": int((gt[has].to_numpy() == other.to_numpy()).sum()),\n                     "збігається після нормалізації": int((gnn.to_numpy() == on.to_numpy()).sum()),\n                     "збігаються перші 60 символів після нормалізації": int((gnn.str[:60].to_numpy() == on.str[:60].to_numpy()).sum()),\n                     "медіана довжини": float(other.str.len().median())})\n    summary["gold_text_median_length"] = float(gt.str.len().median())\n    tables["text_variants"] = pd.DataFrame(rows)\n    # копії золотих текстів у корпусі (для виключення в P1)\n    norm_all = normalize_text(msgs["text"])\n    gold_norm = set(gn[gn != ""])\n    is_copy = norm_all.isin(gold_norm).to_numpy()\n    not_self = ~keys(msgs).isin(set(km_gold)).to_numpy()\n    summary["n_corpus_copies_of_gold_text"] = int((is_copy & not_self).sum())\n    gcopy = gn.isin(set(norm_all[is_copy & not_self]))\n    summary["n_gold_with_corpus_copy"] = int(gcopy.sum())\n    dup = norm_all.duplicated(keep=False)\n    labd = _flag(msgs["is_labeled_batch"])\n    summary["duplicates"] = {"n_rows_in_duplicate_groups": int(dup.sum()), "share_all": float(dup.mean()),\n                             "share_labeled": float(dup[labd.to_numpy()].mean()), "n_unique_texts": int(norm_all.nunique())}\n    return summary, tables\n\n\n# ---------------------------------------------------------------- запуск\n\ndef run_all(inp: dict, lang_fn, lang_sample_unlabeled: int = 150_000, seed: int = 42,\n            gcols: dict = GOLD_COLS) -> tuple[dict, dict, list]:\n    msgs, tg, channels, coding = inp["msgs"], inp["tg"], inp["channels"], inp["coding"]\n    gold = inp["gold"]\n    tables: dict[str, pd.DataFrame] = {}\n    summary: dict = {"readme_counts": README_COUNTS}\n    summary["integrity"] = check_integrity(msgs, tg)\n    tables["monthly"] = monthly_table(msgs)\n    tables["per_channel"] = per_channel(msgs, tg, channels)\n    s, t = label_tables(msgs, coding)\n    summary["labels"] = s\n    tables.update(t)\n    tables.update(language_tables(msgs, gold, lang_fn, lang_sample_unlabeled, seed, gcols))\n    tables["graph_membership"] = graph_membership(msgs, inp["fwd"], inp["sem"])\n    summary["forwards"] = forwards_summary(tg, msgs, channels, len(inp["fwd"]))\n    gs, gt = gold_overlap(gold, msgs, coding, inp["tg_gold_text"], inp["fwd"], inp["sem"], gcols)\n    summary["gold"] = gs\n    tables.update(gt)\n    gb = gold[gcols["binary"]].astype(bool)\n    summary["gold_narrative_share"] = float(gb.mean())\n    gm = gold.loc[gb, gcols["level1"]].astype(str).value_counts()\n    tables["gold_meta_counts"] = pd.DataFrame({"level_1": gm.index, "n_gold": gm.to_numpy(),\n                                               "share_gold": gm.to_numpy() / gm.sum()})\n    summary["edges"] = {"forward_edges": int(len(inp["fwd"])), "semantic_edges": int(len(inp["sem"])),\n                        "semantic_min_similarity": float(inp["sem"]["similarity_score"].min())}\n    return tables, summary, run_checks(tables, summary)\n\n\ndef run_checks(tables: dict, summary: dict) -> list[dict]:\n    """hard=True: інваріанти (дублікати ключів, непарсовані дати), їх порушення ламає наступні етапи.\n    hard=False: очікування з README і плану, розбіжність не блокує етап, а потрапляє у звіт як знахідка."""\n    it, lb, rc = summary["integrity"], summary["labels"], summary["readme_counts"]\n    pc = tables["per_channel"]\n    pc_msgs = pc[pc["n_messages"] > 0]\n    checks = [\n        (True, "Немає дублікатів (channel_id, message_id) у messages", it["dup_keys_msgs"] == 0, f"дублікатів {it[\'dup_keys_msgs\']}"),\n        (True, "Немає дублікатів (channel_id, message_id) у telegram_messages", it["dup_keys_tg"] == 0, f"дублікатів {it[\'dup_keys_tg\']}"),\n        (True, "Усі дати розпарсені", it["unparsed_dates_msgs"] == 0 and it["unparsed_dates_tg"] == 0,\n         f"непарсованих {it[\'unparsed_dates_msgs\']} і {it[\'unparsed_dates_tg\']}"),\n        (False, f"Дати в межах {DATE_MIN} .. {DATE_MAX}", it["out_of_range_msgs"] == 0 and it["out_of_range_tg"] == 0,\n         f"поза межами {it[\'out_of_range_msgs\']} (messages), {it[\'out_of_range_tg\']} (telegram_messages)"),\n        (False, "Наративні повідомлення входять до розмічених", lb["narrative_not_labeled"] == 0, f"поза розміченими {lb[\'narrative_not_labeled\']}"),\n        (False, "sub_narrative_id у мітках відомі файлу кодування", lb["sub_ids_not_in_coding"] == 0, f"невідомих {lb[\'sub_ids_not_in_coding\']}"),\n        (False, "Кожен sub_narrative_id веде до того самого narrative_id, що й у мітці", lb["sub_to_narrative_match_share"] == 1.0,\n         f"збіг {lb[\'sub_to_narrative_match_share\']:.4f}"),\n        (False, "Кількість повідомлень збігається з README", it["n_msgs"] == rc["messages"], f"{it[\'n_msgs\']} проти {rc[\'messages\']}"),\n        (False, "Кількість каналів у таблиці каналів збігається з README", len(pc) == rc["channels"], f"{len(pc)} проти {rc[\'channels\']}"),\n        (False, "Кількості ребер збігаються з README",\n         summary["edges"]["forward_edges"] == rc["forward_edges"] and summary["edges"]["semantic_edges"] == rc["semantic_edges"],\n         f"forward {summary[\'edges\'][\'forward_edges\']}, semantic {summary[\'edges\'][\'semantic_edges\']}"),\n        (False, "Частка наративних серед розмічених близька до золотої (різниця до 0,10)",\n         abs(lb["narrative_share_of_labeled"] - summary["gold_narrative_share"]) <= 0.10,\n         f"слабкі {lb[\'narrative_share_of_labeled\']:.3f}, золота {summary[\'gold_narrative_share\']:.3f}"),\n        (False, "Мітки каналів у таблиці збігаються з перерахунком за повідомленнями", bool(pc_msgs["counts_match"].all()),\n         f"збігається {int(pc_msgs[\'counts_match\'].sum())} з {len(pc_msgs)} каналів"),\n        (False, "Усі золоті повідомлення знайдені в messages", summary["gold"]["n_gold_in_graph_messages"] == summary["gold"]["n_gold"],\n         f"{summary[\'gold\'][\'n_gold_in_graph_messages\']} з {summary[\'gold\'][\'n_gold\']}"),\n    ]\n    return [{"hard": h, "check": n, "passed": bool(ok), "detail": d} for h, n, ok, d in checks]\n\n\n# ---------------------------------------------------------------- збереження\n\ndef _default(o):\n    if isinstance(o, (np.integer,)):\n        return int(o)\n    if isinstance(o, (np.floating,)):\n        return None if np.isnan(o) else float(o)\n    if isinstance(o, (np.bool_,)):\n        return bool(o)\n    if isinstance(o, (pd.Timestamp,)):\n        return str(o)\n    raise TypeError(type(o))\n\n\ndef save_all(tables: dict, summary: dict, checks: list, out_dir: Path | str) -> list[Path]:\n    out_dir = Path(out_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n    paths = []\n    for name, df in tables.items():\n        p = out_dir / f"{name}.csv"\n        df.to_csv(p, index=False)\n        paths.append(p)\n    (out_dir / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2, default=_default), encoding="utf-8")\n    (out_dir / "checks.json").write_text(json.dumps(checks, ensure_ascii=False, indent=2), encoding="utf-8")\n    return paths + [out_dir / "summary.json", out_dir / "checks.json"]\n\n\n# ---------------------------------------------------------------- графіки\n\ndef make_figures(tables: dict, summary: dict, out_dir: Path | str) -> list[Path]:\n    import matplotlib\n    matplotlib.use("Agg")\n    import matplotlib.pyplot as plt\n\n    out_dir = Path(out_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n    plt.rcParams.update({"figure.dpi": 150, "savefig.dpi": 150, "axes.spines.top": False, "axes.spines.right": False,\n                         "font.size": 9})\n    paths = []\n\n    def save(fig, name):\n        p = out_dir / name\n        fig.savefig(p, bbox_inches="tight")\n        plt.close(fig)\n        paths.append(p)\n\n    # 1. Повідомлення за місяцями\n    mo = tables["monthly"]\n    fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))\n    colors = np.where(mo["partial_month"], "#b8b8b8", "#3b6fb6")\n    ax[0].bar(mo["month"], mo["n"] / 1000, color=colors)\n    ax[0].set_ylabel("повідомлень, тис.")\n    ax[0].set_title("Усі повідомлення за місяцями (сірі: неповні місяці)")\n    ax[1].bar(mo["month"], mo["n_labeled"] / 1000, color="#8aa9d6", label="розмічені")\n    ax[1].bar(mo["month"], mo["n_narrative"] / 1000, color="#c0392b", label="з наративом")\n    ax[1].set_ylabel("повідомлень, тис.")\n    ax[1].set_title("Слабкі мітки за місяцями")\n    ax[1].legend(frameon=False)\n    for a in ax:\n        a.tick_params(axis="x", rotation=70)\n    save(fig, "01_monthly.png")\n\n    # 2. Канали: обсяг і частка наративних\n    pc = tables["per_channel"]\n    top = pc[pc["n_messages"] > 0].head(25).iloc[::-1]\n    fig, ax = plt.subplots(1, 2, figsize=(11, 6), sharey=True)\n    ax[0].barh(top["username"], top["n_messages"] / 1000, color="#3b6fb6")\n    ax[0].set_xlabel("повідомлень, тис.")\n    ax[0].set_title("25 найбільших каналів")\n    ax[1].barh(top["username"], top["share_recomputed"].fillna(0), color="#c0392b")\n    ax[1].set_xlabel("частка наративних серед розмічених")\n    ax[1].set_xlim(0, 1)\n    ax[1].set_title("Слабкі мітки")\n    save(fig, "02_channels.png")\n\n    # 3. Мова і частка наративних\n    lx = tables["language_x_label"]\n    fig, ax = plt.subplots(figsize=(6.5, 3.8))\n    langs = sorted(lx["lang"].unique())\n    w = 0.38\n    for i, (src, col) in enumerate((("слабкі мітки (розмічені)", "#8aa9d6"), ("золота розмітка", "#c0392b"))):\n        d = lx[lx["source"] == src].set_index("lang").reindex(langs)\n        xs = np.arange(len(langs)) + (i - 0.5) * w\n        ax.bar(xs, d["narrative_share"].fillna(0), w, label=src, color=col)\n        for x, (_, r) in zip(xs, d.iterrows()):\n            if pd.notna(r["n"]):\n                ax.text(x, (r["narrative_share"] if pd.notna(r["narrative_share"]) else 0) + 0.01, f"n={int(r[\'n\'])}",\n                        ha="center", fontsize=7)\n    ax.set_xticks(np.arange(len(langs)))\n    ax.set_xticklabels(langs)\n    ax.set_ylabel("частка наративних")\n    ax.set_ylim(0, 1.1)\n    ax.set_title("Мова і частка наративних повідомлень")\n    ax.legend(frameon=False)\n    save(fig, "03_language.png")\n\n    # 4. Мета-наративи: слабкі мітки проти золотої розмітки\n    meta = tables["meta_weak"].copy()\n    gold_meta = tables.get("gold_meta_counts")\n    fig, ax = plt.subplots(figsize=(9, 4.2))\n    meta = meta.sort_values("n_weak")\n    ax.barh(meta["level_1"].astype(str).str.slice(0, 48), meta["share_weak"], color="#8aa9d6", label="слабкі мітки")\n    if gold_meta is not None:\n        gm = gold_meta.set_index("level_1")["share_gold"]\n        ax.scatter(meta["level_1"].map(gm).fillna(0), range(len(meta)), color="#c0392b", zorder=3, label="золота розмітка")\n    ax.set_xlabel("частка серед наративних")\n    ax.set_title("Розподіл мета-наративів")\n    ax.legend(frameon=False)\n    save(fig, "04_meta.png")\n\n    # 5. Довжина тексту\n    ls = tables["length_stats"].set_index("group")\n    fig, ax = plt.subplots(figsize=(6.5, 3.6))\n    cols = ["розмічені, ненаративні", "розмічені, наративні", "нерозмічені"]\n    ax.bar(range(len(cols)), [ls.loc[c, "median"] for c in cols], color=["#8aa9d6", "#c0392b", "#b8b8b8"])\n    ax.errorbar(range(len(cols)), [ls.loc[c, "median"] for c in cols],\n                yerr=[[ls.loc[c, "median"] - ls.loc[c, "p10"] for c in cols], [ls.loc[c, "p90"] - ls.loc[c, "median"] for c in cols]],\n                fmt="none", color="k", capsize=4)\n    ax.set_xticks(range(len(cols)))\n    ax.set_xticklabels(cols, fontsize=8)\n    ax.set_ylabel("символів (медіана, діапазон p10-p90)")\n    ax.set_title("Довжина тексту")\n    save(fig, "05_length.png")\n\n    # 6. Упевненість міток і слабкі мітки проти золотих\n    cf = tables["label_confidence"]\n    fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))\n    ax[0].bar(cf["label_confidence"].astype(str), cf["n_not_narrative"], label="без наративу", color="#8aa9d6")\n    ax[0].bar(cf["label_confidence"].astype(str), cf["n_narrative"], bottom=cf["n_not_narrative"], label="з наративом", color="#c0392b")\n    ax[0].set_xlabel("label_confidence")\n    ax[0].set_ylabel("розмічених повідомлень")\n    ax[0].set_title("Упевненість слабких міток")\n    ax[0].tick_params(axis="x", rotation=60)\n    ax[0].legend(frameon=False)\n    cm = tables.get("weak_vs_gold_confusion")\n    if cm is not None:\n        arr = cm.iloc[:, 1:].to_numpy(dtype=float)\n        ax[1].imshow(arr, cmap="Blues")\n        for i in range(2):\n            for j in range(2):\n                ax[1].text(j, i, f"{int(arr[i, j])}", ha="center", va="center", color="k")\n        ax[1].set_xticks([0, 1])\n        ax[1].set_xticklabels(["слабка: ні", "слабка: так"])\n        ax[1].set_yticks([0, 1])\n        ax[1].set_yticklabels(["золота: ні", "золота: так"])\n        ax[1].set_title("Слабкі мітки проти золотих (де є обидві)")\n    save(fig, "06_confidence_and_agreement.png")\n    return paths\n', 'src/narr/language.py': '"""Мова повідомлення (uk / ru / en) через langid. Евристика: на коротких текстах можливі помилки."""\nfrom __future__ import annotations\n\nimport numpy as np\n\n\ndef detect_language(texts, langs: tuple[str, ...] = ("uk", "ru", "en")) -> np.ndarray:\n    import langid\n\n    langid.set_languages(list(langs))\n    return np.array([langid.classify(str(t))[0] for t in texts])\n\n\ndef _detect_chunk(texts: list[str]) -> list[str]:\n    return list(detect_language(texts))\n\n\ndef detect_language_parallel(texts, n_jobs: int = 2, min_per_job: int = 2000) -> np.ndarray:\n    """Те саме, що detect_language, але частинами в кількох процесах. Порядок збережено.\n    Для малих вхідних масивів або n_jobs <= 1 працює послідовно."""\n    texts = [str(t) for t in texts]\n    if n_jobs <= 1 or len(texts) < n_jobs * min_per_job:\n        return detect_language(texts)\n    from multiprocessing import get_context\n\n    chunks = [c.tolist() for c in np.array_split(np.array(texts, dtype=object), n_jobs * 8) if len(c)]\n    with get_context("fork").Pool(n_jobs) as pool:\n        parts = pool.map(_detect_chunk, chunks)\n    return np.array([x for part in parts for x in part])\n', 'src/narr/io.py': '"""Конфігурація, завантаження даних, валідація схем, службова інформація про запуск."""\nfrom __future__ import annotations\n\nimport datetime as dt\nimport json\nimport platform\nimport subprocess\nfrom pathlib import Path\n\nimport pandas as pd\nimport yaml\n\nROOT = Path(__file__).resolve().parents[2]\nCONFIG_PATH = ROOT / "configs" / "base.yaml"\n\n\ndef load_config(path: Path | str | None = None) -> dict:\n    with open(path or CONFIG_PATH, encoding="utf-8") as f:\n        return yaml.safe_load(f)\n\n\ndef path_of(cfg: dict, key: str) -> Path:\n    return ROOT / cfg["paths"][key]\n\n\ndef data_available(cfg: dict | None = None) -> bool:\n    cfg = cfg or load_config()\n    return path_of(cfg, "golden").exists()\n\n\ndef load_golden(cfg: dict | None = None) -> pd.DataFrame:\n    cfg = cfg or load_config()\n    df = pd.read_csv(path_of(cfg, "golden"))\n    df[cfg["columns"]["date"]] = pd.to_datetime(df[cfg["columns"]["date"]], utc=True)\n    return df\n\n\ndef load_shared(cfg: dict | None = None) -> pd.DataFrame:\n    cfg = cfg or load_config()\n    return pd.read_csv(path_of(cfg, "shared"))\n\n\ndef validate_golden(df: pd.DataFrame, cfg: dict | None = None) -> list[str]:\n    """Повертає список проблем. Порожній список означає, що схема і вміст відповідають очікуванням."""\n    cfg = cfg or load_config()\n    col, exp = cfg["columns"], cfg["expected"]\n    no_narr = col["no_narrative_label"]\n    problems: list[str] = []\n\n    required = [col["text"], col["channel"], col["message"], col["date"], col["forwarded"],\n                col["binary"], col["general"], col["level1"], *col["reviewers"], *col["reviewers_level1"]]\n    missing = [c for c in required if c not in df.columns]\n    if missing:\n        return [f"відсутні колонки: {missing}"]\n\n    if len(df) != exp["n_messages"]:\n        problems.append(f"рядків {len(df)}, очікувалось {exp[\'n_messages\']}")\n    n_dup = int(df.duplicated([col["channel"], col["message"]]).sum())\n    if n_dup:\n        problems.append(f"дублікатів (channel_id, message_id): {n_dup}")\n    n_pos = int(df[col["binary"]].sum())\n    if n_pos != exp["n_narrative"]:\n        problems.append(f"наративних {n_pos}, очікувалось {exp[\'n_narrative\']}")\n    if int(df[col["channel"]].nunique()) != exp["n_channels_in_gold"]:\n        problems.append(f"каналів у вибірці {df[col[\'channel\']].nunique()}, очікувалось {exp[\'n_channels_in_gold\']}")\n    if df[col["text"]].isna().any() or (df[col["text"]].astype(str).str.strip() == "").any():\n        problems.append("є порожні тексти")\n    if df[col["date"]].isna().any():\n        problems.append("є непарсовані дати")\n    if ((df[col["general"]] != no_narr) != df[col["binary"]]).any():\n        problems.append("Binary label не узгоджена з General label")\n    if ((df[col["level1"]] != no_narr) != df[col["binary"]]).any():\n        problems.append("Binary label не узгоджена з General label (level-1)")\n    return problems\n\n\ndef _git(*args: str) -> str | None:\n    try:\n        out = subprocess.run(["git", "-C", str(ROOT), *args], capture_output=True, text=True, check=True)\n        return out.stdout.strip()\n    except Exception:\n        return None\n\n\ndef write_run_json(out_dir: Path | str, script: str, params: dict) -> Path:\n    """Пише run.json: дата, git-хеш, параметри, версії бібліотек."""\n    import numpy\n    import sklearn\n\n    out_dir = Path(out_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n    info = {\n        "script": script,\n        "timestamp_utc": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),\n        "git_commit": _git("rev-parse", "HEAD"),\n        "git_dirty": bool(_git("status", "--porcelain", "--untracked-files=no", "--", "src", "scripts", "configs")),  # зміни коду відносно коміту\n        "params": params,\n        "versions": {\n            "python": platform.python_version(),\n            "numpy": numpy.__version__,\n            "pandas": pd.__version__,\n            "scikit-learn": sklearn.__version__,\n        },\n    }\n    path = out_dir / "run.json"\n    path.write_text(json.dumps(info, ensure_ascii=False, indent=2), encoding="utf-8")\n    return path\n', 'configs/base.yaml': '# Базова конфігурація проєкту. Усі шляхи відносно кореня репозиторію.\nseed: 42\n\npaths:\n  telenarratives: external/TeleNarratives\n  golden: "external/TeleNarratives/labeling results/strong labeling/golden set (labeled).csv"\n  shared: "external/TeleNarratives/labeling results/strong labeling/shared subset (labeled).csv"\n  iaa: "external/TeleNarratives/labeling results/strong labeling/inter_annotator_agreement_metrics.csv"\n  channels: external/TeleNarratives/config/channels.csv\n  narratives: external/TeleNarratives/config/narratives.csv\n  important_dates: "external/TeleNarratives/config/Important dates.csv"\n  gnn_results: "external/TeleNarratives/gnn results"\n  weak_eval: "external/TeleNarratives/labeling results/weak labeling/evaluation"\n\ncolumns:\n  no_narrative_label: "Не містить наративу"\n  binary: "Binary label"\n  general: "General label"\n  level1: "General label (level-1)"\n  text: text\n  channel: channel_id\n  message: message_id\n  date: date_utc\n  forwarded: is_forwarded\n  reviewers: ["Reviewer 1", "Reviewer 2", "Reviewer 3"]\n  # у файлі третій рецензент на рівні level-1 названий "Reviewer 2 (level-1).1"\n  reviewers_level1: ["Reviewer 1 (level-1)", "Reviewer 2 (level-1)", "Reviewer 2 (level-1).1"]\n\n# Очікувані значення з PLAN_vykonannya.md, розділ 2.2. Перевіряються тестами і аудитом.\nexpected:\n  n_messages: 412\n  n_narrative: 148\n  n_shared: 103\n  n_channels_config: 98\n  n_sub_narratives: 380\n  n_channels_in_gold: 56\n\neval:\n  bootstrap_n: 1000\n  cv_folds: 5\n  cv_repeats: 5\n  n_top_meta_narratives: 4   # багатокласова оцінка: none + 4 найбільші + «інше»\n  threshold: 0.5\n', 'scripts/s02_eda.py': '"""S2: описова аналітика повних даних TeleNarratives (дані з OSF, знімок графу 11.03.2026).\n\nЗапуск:  python3 -I scripts/s02_eda.py --raw-dir <data/raw> [--out results/02_eda] [--lang-sample 150000]\nДані читаються з parquet-набору (messages, channels, forward_edges, semantic_edges, кодування наративів)\nі з telegram_messages.parquet. Золотий набір береться за шляхом з configs/base.yaml.\nРезультат: CSV-таблиці, summary.json, checks.json, run.json, figures/*.png.\nКод завершується з ненульовим статусом, якщо порушено хоч один жорсткий інваріант (hard=True у checks.json).\n"""\nimport argparse\nimport sys\nimport time\nfrom pathlib import Path\n\nsys.path.insert(0, str(Path(__file__).resolve().parents[1] / "src"))\n\nfrom narr import eda  # noqa: E402\nfrom narr.io import ROOT, load_config, path_of, validate_golden, write_run_json  # noqa: E402\nfrom narr.language import detect_language_parallel  # noqa: E402\n\n\ndef main(argv=None) -> int:\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--raw-dir", required=True, help="папка data/raw (з telegram_messages.parquet і розпакованим parquet-набором)")\n    ap.add_argument("--out", default=None, help="папка результатів (типово results/02_eda у корені репозиторію)")\n    ap.add_argument("--golden", default=None, help="шлях до golden set CSV (типово з configs/base.yaml)")\n    ap.add_argument("--lang-sample", type=int, default=150_000, help="розмір вибірки нерозмічених повідомлень для визначення мови")\n    ap.add_argument("--jobs", type=int, default=2)\n    ap.add_argument("--no-validate-gold", action="store_true", help="лише для тестів на синтетичних даних")\n    args = ap.parse_args(argv)\n\n    cfg = load_config()\n    seed = cfg["seed"]\n    out = Path(args.out) if args.out else ROOT / "results" / "02_eda"\n    golden = Path(args.golden) if args.golden else path_of(cfg, "golden")\n\n    t0 = time.time()\n    print("Читаю дані ...", flush=True)\n    inp = eda.load_inputs(args.raw_dir, golden)\n    print(f"  messages {len(inp[\'msgs\']):,}, telegram_messages {len(inp[\'tg\']):,}, золотих {len(inp[\'gold\'])}, "\n          f"{time.time() - t0:.0f} с", flush=True)\n\n    print("Рахую таблиці і мову ...", flush=True)\n    lang_fn = lambda s: detect_language_parallel(s, n_jobs=args.jobs)  # noqa: E731\n    tables, summary, checks = eda.run_all(inp, lang_fn, lang_sample_unlabeled=args.lang_sample, seed=seed)\n\n    if not args.no_validate_gold:\n        problems = validate_golden(inp["gold"], cfg)\n        checks.append({"hard": True, "check": "Золотий набір проходить validate_golden", "passed": not problems,\n                       "detail": "; ".join(problems) if problems else "412 рядків, 148 наративних, 56 каналів, мітки узгоджені"})\n\n    eda.save_all(tables, summary, checks, out)\n    figs = eda.make_figures(tables, summary, out / "figures")\n    write_run_json(out, "scripts/s02_eda.py", {"raw_dir": str(args.raw_dir), "lang_sample": args.lang_sample, "seed": seed,\n                                               "golden": str(golden), "seconds": round(time.time() - t0)})\n    print(f"Збережено {len(tables)} таблиць і {len(figs)} графіків у {out}", flush=True)\n\n    bad = [c for c in checks if c["hard"] and not c["passed"]]\n    for c in checks:\n        mark = "ok  " if c["passed"] else ("ПОМИЛКА" if c["hard"] else "розбіжність")\n        print(f"[{mark}] {c[\'check\']}: {c[\'detail\']}")\n    if bad:\n        print(f"\\nЖорстких порушень: {len(bad)}")\n        return 1\n    return 0\n\n\nif __name__ == "__main__":\n    sys.exit(main())\n'}

for rel, text in FILES.items():
    p = REPO / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(text, encoding="utf-8")
    print(f"{hashlib.sha256(text.encode()).hexdigest()[:12]}  {rel}")


In [ ]:
# 3. Золотий набір: клон TeleNarratives (потрібен лише golden set CSV)
EXT = REPO / "external" / "TeleNarratives"
if not EXT.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/yuliavistak/TeleNarratives", str(EXT)], check=True)
head = subprocess.run(["git", "-C", str(EXT), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("TeleNarratives commit:", head)
gold_csv = next(EXT.glob("labeling results/strong labeling/golden set (labeled).csv"))
print("Золотий набір:", gold_csv)


In [ ]:
# 4. Перевірка, що потрібні файли на місці
need = ["telegram_messages.parquet", "narratives_hierarchy.csv"]
for n in need:
    assert (RAW / n).exists(), f"немає {n}"
msgs_p = sorted(RAW.rglob("messages.parquet"))
assert msgs_p, "немає messages.parquet у розпакованому parquet-наборі"
print("messages.parquet:", msgs_p[0])
try:
    import psutil
    print(f"Пам'ять: {psutil.virtual_memory().total / 1e9:.1f} ГБ усього, {psutil.virtual_memory().available / 1e9:.1f} ГБ вільно")
except ImportError:
    pass


In [ ]:
# 5. Запуск S2 (скрипт scripts/s02_eda.py з репозиторію)
cmd = [sys.executable, "-I", str(REPO / "scripts" / "s02_eda.py"), "--raw-dir", str(RAW), "--out", str(OUT),
       "--lang-sample", "150000", "--jobs", "2"]
t0 = time.time()
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=str(REPO))
for line in proc.stdout:
    print(line, end="")
code = proc.wait()
print(f"\nКод завершення {code}, {time.time() - t0:.0f} с")


In [ ]:
# 6. Підсумок, один JSON з усіма таблицями і архів
import shutil
import pandas as pd

summary = json.loads((OUT / "summary.json").read_text(encoding="utf-8"))
checks = json.loads((OUT / "checks.json").read_text(encoding="utf-8"))

tables = {}
for p in sorted(OUT.glob("*.csv")):
    df = pd.read_csv(p)
    tables[p.stem] = df.to_dict(orient="records")
(OUT / "tables_all.json").write_text(json.dumps(tables, ensure_ascii=False, default=str), encoding="utf-8")

shutil.make_archive(str(OUT.parent / "02_eda"), "zip", OUT)
print("Архів:", OUT.parent / "02_eda.zip")

print("\nПеревірки:")
for c in checks:
    mark = "ok" if c["passed"] else ("ПОМИЛКА" if c["hard"] else "розбіжність")
    print(f"  [{mark}] {c['check']}: {c['detail']}")
print("\nФайли в", OUT)
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(OUT)}  {p.stat().st_size / 1e3:.1f} КБ")
assert code == 0, "S2: жорсткі перевірки не пройдено, див. вивід вище"
